## Automatic Question Generating System
Name: Siddhi Totala, Tanvi Parakh


In [15]:
!pip install streamlit pyngrok pdfplumber nltk spacy transformers sentencepiece
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 71.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [16]:
import nltk
import spacy

nltk.download('punkt')
nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [27]:
!ngrok config add-authtoken 3BTXl1NZmdqegx9apANUtW0vE1f_3X1pH6WT71JsWUf8Fc2TY

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [40]:
%%writefile app.py

import streamlit as st
import pdfplumber
import nltk
import spacy
import random
from transformers import T5ForConditionalGeneration, T5Tokenizer

# Setup
nltk.download('punkt')
nltk.download('punkt_tab') # Added to resolve the LookupError
nlp = spacy.load("en_core_web_sm")

# Model
model_name = "valhalla/t5-small-qg-hl"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

st.set_page_config(page_title="AQG System", layout="wide")

st.title("📚 Automatic Question Generator")

uploaded_file = st.file_uploader("Upload PDF", type="pdf")

# -------- TEXT --------
def extract_text(file):
    text = ""
    with pdfplumber.open(file) as pdf:
        for page in pdf.pages:
            content = page.extract_text()
            if content:
                text += content + "\n"
    return text
def clean_sentences(text):
    sents = nltk.sent_tokenize(text)
    clean = []

    for s in sents:
        if len(s.split()) < 8:
            continue
        if any(char.isdigit() for char in s):
            continue
        if len(s) < 50:
            continue
        if "_" in s:
            continue

        clean.append(s)

    return clean[:6]

# -------- FILL BLANK --------
def generate_fill_blanks(sentences):
    questions = []
    answers = []

    for sent in sentences:
        doc = nlp(sent)

        for ent in doc.ents:
            # Only meaningful entities
            if ent.label_ in ["ORG", "PERSON", "GPE", "DATE", "EVENT"]:

                q = sent.replace(ent.text, "_____")

                if len(q.split()) >= 6:
                    questions.append(q)
                    answers.append(ent.text)
                    break

    return questions, answers

# -------- MCQ --------
def generate_mcq(sentences):
    mcqs = []

    for sent in sentences:
        doc = nlp(sent)

        for ent in doc.ents:
            correct = ent.text

            # collect similar entities
            distractors = [e.text for e in doc.ents if e.text != correct]

            # fallback smart options
            fallback = [
                "All of the above",
                "None of the above",
                "Cannot be determined",
                "Not mentioned in the text"
            ]

            i = 0
            while len(distractors) < 3:
                distractors.append(fallback[i])
                i += 1

            options = [correct] + distractors[:3]
            random.shuffle(options)

            question = sent.replace(ent.text, "_____")

            if len(question.split()) >= 6:
                mcqs.append((question, options, correct))
                break

    return mcqs

# -------- WH --------
def generate_wh(sentence):
    sentence = sentence.replace("_", "").strip()

    doc = nlp(sentence)

    for ent in doc.ents:
        if ent.label_ in ["ORG", "PERSON", "GPE", "DATE", "EVENT"]:

            highlighted = sentence.replace(ent.text, f"<hl> {ent.text} <hl>")

            input_text = "generate question: " + highlighted

            input_ids = tokenizer.encode(input_text, return_tensors="pt")

            output = model.generate(
                input_ids,
                max_length=80,
                num_beams=5,
                early_stopping=True
            )

            question = tokenizer.decode(output[0], skip_special_tokens=True)

            # Quality filter
            if len(question.split()) >= 5:
                return question

    return None

# -------- MAIN --------
if uploaded_file:

    if st.button("Generate"):

        text = extract_text(uploaded_file)

        if not text.strip():
            st.error("No text found ❌")
            st.stop()

        # Limit text for speed
        text = text[:2000]

        # Clean sentences (IMPORTANT)
        sentences = clean_sentences(text)

        # =========================
        # WH QUESTIONS (10)
        # =========================
        wh = [generate_wh(s) for s in sentences]
        wh = [q for q in wh if q]

        while len(wh) < 10 and len(wh) > 0:
            wh.append(wh[len(wh) % len(wh)])

        wh = wh[:10]

        # =========================
        # FILL IN THE BLANKS (10)
        # =========================
        fb_q, fb_a = generate_fill_blanks(sentences)

        while len(fb_q) < 10 and len(fb_q) > 0:
            fb_q.append(fb_q[len(fb_q) % len(fb_q)])
            fb_a.append(fb_a[len(fb_a) % len(fb_a)])

        fb_q, fb_a = fb_q[:10], fb_a[:10]

        # =========================
        # MCQ (10)
        # =========================
        mcqs = generate_mcq(sentences)

        while len(mcqs) < 10 and len(mcqs) > 0:
            mcqs.append(mcqs[len(mcqs) % len(mcqs)])

        mcqs = mcqs[:10]

        # =========================
        # STORE IN SESSION
        # =========================
        st.session_state.fb_q = fb_q
        st.session_state.fb_a = fb_a
        st.session_state.mcqs = mcqs
        st.session_state.wh = wh

# -------- MODES --------
if "fb_q" in st.session_state:

    mode = st.radio("Mode", ["AI Questions", "Quiz", "Fill Blanks", "Flashcards"])

    fb_q = st.session_state.fb_q
    fb_a = st.session_state.fb_a
    mcqs = st.session_state.mcqs
    wh = st.session_state.wh

    if mode == "AI Questions":
        for q in wh:
            st.write(q)

    elif mode == "Fill Blanks":
        for q, a in zip(fb_q, fb_a):
            st.write(q)
            st.success(a)

    elif mode == "Quiz":

        user_ans = {}

        for i, (q, opts, ans) in enumerate(mcqs):
            user_ans[i] = st.radio(q, opts, key=i)

        if st.button("Submit"):
            score = 0
            for i, (q, opts, ans) in enumerate(mcqs):
                if user_ans[i] == ans:
                    score += 1
            st.success(f"Score: {score}/{len(mcqs)}")

    elif mode == "Flashcards":

        if "i" not in st.session_state:
            st.session_state.i = 0

        i = st.session_state.i

        st.write(fb_q[i])

        if st.button("Show Answer"):
            st.success(fb_a[i])

        if st.button("Next"):
            st.session_state.i = min(len(fb_q)-1, i+1)

        if st.button("Prev"):
            st.session_state.i = max(0, i-1)

Overwriting app.py


In [41]:
!streamlit run app.py &>/dev/null &

In [42]:
from pyngrok import ngrok
ngrok.kill()

In [43]:
public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://excogitative-keli-triradiately.ngrok-free.dev" -> "http://localhost:8501"
